# 检索

大型语言模型（LLM）功能强大，但存在两个主要局限性：
- 上下文有限——他们无法一次性摄取整个语料库。
- 静态知识——他们的训练数据被冻结在某个时间点。

检索通过在查询时获取相关的外部知识来解决这些问题。这就是检索增强生成（RAG）的基础：利用上下文信息增强语言学习模型（LLM）的答案。

## 构建知识库
知识库是用于检索过程中的文档或结构化数据的存储库。

如果您需要自定义知识库，可以使用 LangChain 的文档加载器和矢量存储，根据您自己的数据构建一个知识库。

请参阅以下教程，了解如何构建可搜索的知识库和最简化的 RAG 工作流程：

## 从检索到 RAG
检索功能使语言学习模型能够在运行时访问相关上下文。但大多数实际应用更进一步：它们将检索与生成相结合，以生成基于上下文的、有理有据的答案。
这就是检索增强生成（RAG）的核心思想。检索流程成为一个更广泛的系统的基础，该系统将搜索与生成相结合。
​
## 检索管道
典型的检索工作流程如下所示：

<img src="https://i-blog.csdnimg.cn/direct/ed1d5178ff394a14859b1479163d672f.png" >

每个组件都是模块化的：您可以交换加载器、分割器、嵌入或矢量存储，而无需重写应用程序的逻辑。

## RAG架构
RAG 可以通过多种方式实现，具体取决于您的系统需求。我们将在以下章节中详细介绍每种方式。

| 建筑学 | 描述 | 控制 | 灵活性 | 延迟 |
| :--- | :--- | :--- | :--- | :--- |
| 2-Step RAG | 检索总是在生成之前发生。简单且可预测。 | 高 |  | 快速 |
| Agentic RAG | 基于LLM的智能体在推理过程中决定何时以及如何检索信息。 | 低 | 高 | 可变 |
| Hybrid | 结合两种方法的特点，并加入验证步骤 |中等 | 工中等 | 可变 |

## 两步 RAG
在两步 RAG 算法中，检索步骤始终在生成步骤之前执行。这种架构简单明了且可预测，因此适用于许多应用场景，在这些场景中，检索相关文档是生成答案的明确前提。

<img src="https://i-blog.csdnimg.cn/direct/07513d601dfb40a89c7dac27530afb92.png">

## Agentic RAG
智能检索增强生成（RAG）结合了检索增强生成和基于智能体的推理的优势。它不是在回答问题之前检索文档，而是由智能体（由逻辑逻辑模型驱动）逐步推理，并在交互过程中决定何时以及如何检索信息。

<img src="https://i-blog.csdnimg.cn/direct/81a0a688831642669793c4a3c63b5057.png">

In [ ]:
import requests
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent


@tool
def fetch_url(url: str) -> str:
    """Fetch text content from a URL"""
    response = requests.get(url, timeout=10.0)
    response.raise_for_status()
    return response.text

system_prompt = """\
Use fetch_url when you need to fetch information from a web-page; quote relevant snippets.
"""

agent = create_agent(
    model="claude-sonnet-4-5-20250929",
    tools=[fetch_url], # A tool for retrieval
    system_prompt=system_prompt,
)

### 扩展

In [ ]:
import requests
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool
from markdownify import markdownify


ALLOWED_DOMAINS = ["https://langchain-ai.github.io/"]
LLMS_TXT = 'https://langchain-ai.github.io/langgraph/llms.txt'


@tool
def fetch_documentation(url: str) -> str:  
    """Fetch and convert documentation from a URL"""
    if not any(url.startswith(domain) for domain in ALLOWED_DOMAINS):
        return (
            "Error: URL not allowed. "
            f"Must start with one of: {', '.join(ALLOWED_DOMAINS)}"
        )
    response = requests.get(url, timeout=10.0)
    response.raise_for_status()
    return markdownify(response.text)


# We will fetch the content of llms.txt, so this can
# be done ahead of time without requiring an LLM request.
llms_txt_content = requests.get(LLMS_TXT).text

# System prompt for the agent
system_prompt = f"""
You are an expert Python developer and technical assistant.
Your primary role is to help users with questions about LangGraph and related tools.

Instructions:

1. If a user asks a question you're unsure about — or one that likely involves API usage,
   behavior, or configuration — you MUST use the `fetch_documentation` tool to consult the relevant docs.
2. When citing documentation, summarize clearly and include relevant context from the content.
3. Do not use any URLs outside of the allowed domain.
4. If a documentation fetch fails, tell the user and proceed with your best expert understanding.

You can access official documentation from the following approved sources:

{llms_txt_content}

You MUST consult the documentation to get up to date documentation
before answering a user's question about LangGraph.

Your answers should be clear, concise, and technically accurate.
"""

tools = [fetch_documentation]

model = init_chat_model("claude-sonnet-4-0", max_tokens=32_000)

agent = create_agent(
    model=model,
    tools=tools,  
    system_prompt=system_prompt,  
    name="Agentic RAG",
)

response = agent.invoke({
    'messages': [
        HumanMessage(content=(
            "Write a short example of a langgraph agent using the "
            "prebuilt create react agent. the agent should be able "
            "to look up stock pricing information."
        ))
    ]
})

print(response['messages'][-1].content)

## 混合RAG
混合 RAG 结合了两步 RAG 和代理 RAG 的特点。它引入了查询预处理、检索验证和生成后检查等中间步骤。这些系统比固定管道系统更灵活，同时又能对执行过程进行一定的控制。

典型组件包括：
- 查询优化：修改输入的问题以提高检索质量。这可能包括重写不清晰的查询、生成多个变体或使用更多上下文信息扩展查询。
- 检索验证：评估检索到的文档是否相关且充分。如果不符合要求，系统可能会优化查询并重新检索。
- 答案验证：检查生成的答案是否准确、完整，以及是否与原文内容一致。如有需要，系统可以重新生成或修改答案。

该架构通常支持在这些步骤之间进行多次迭代：

<img src="https://i-blog.csdnimg.cn/direct/d62bd151f04748b0a83d4aafc109b5aa.png">

这种架构适用于：
- 包含模糊或不明确查询的应用程序
- 需要验证或质量控制步骤的系统
- 涉及多个数据源或迭代改进的工作流程
